In [1]:
import sys

print("Jupyter Python:")
print(sys.executable)

Jupyter Python:
D:\condaenv\envs\monsoon\python.exe


In [2]:
import sys

!{sys.executable} -m pip install earthaccess

In [3]:
import sys
!{sys.executable} -m pip install -U ipywidgets jupyter

In [4]:
import earthaccess

print("earthaccess imported successfully")
print("Version:", earthaccess.__version__)

earthaccess imported successfully
Version: 0.19.0


In [5]:
from pathlib import Path

import earthaccess
import pandas as pd


PROJECT = Path(r"Z:\Projects\monsoon-postprocessing")

IMERG_PILOT = (
    PROJECT
    / "data"
    / "raw"
    / "imerg_pilot"
)

GEFS_PILOT = (
    PROJECT
    / "data"
    / "raw"
    / "gefs_pilot"
)

IMERG_PILOT.mkdir(parents=True, exist_ok=True)
GEFS_PILOT.mkdir(parents=True, exist_ok=True)

print("IMERG folder:", IMERG_PILOT)
print("GEFS folder:", GEFS_PILOT)

IMERG folder: Z:\Projects\monsoon-postprocessing\data\raw\imerg_pilot
GEFS folder: Z:\Projects\monsoon-postprocessing\data\raw\gefs_pilot


In [6]:
auth = earthaccess.login(
    strategy="interactive",
    persist=True
)

print("Authenticated:", auth.authenticated)

Enter your Earthdata Login username:  arayani
Enter your Earthdata password:  ········


Authenticated: True


In [7]:
imerg_results = earthaccess.search_data(
    short_name="GPM_3IMERGDF",
    version="07",
    temporal=(
        "2018-07-01",
        "2018-07-07"
    ),
    bounding_box=(
        68,
        6,
        98,
        38
    )
)

print("IMERG granules found:", len(imerg_results))

for index, result in enumerate(imerg_results):
    print(index, result)

IMERG granules found: 7
0 Collection: {'ShortName': 'GPM_3IMERGDF', 'Version': '07'}
Spatial coverage: {'HorizontalSpatialDomain': {'Geometry': {'BoundingRectangles': [{'WestBoundingCoordinate': -180.0, 'EastBoundingCoordinate': 180.0, 'NorthBoundingCoordinate': 90.0, 'SouthBoundingCoordinate': -90.0}]}}}
Temporal coverage: {'RangeDateTime': {'BeginningDateTime': '2018-07-01T00:00:00.000Z', 'EndingDateTime': '2018-07-01T23:59:59.999Z'}}
Size(MB): 32.4251613616943
Data: ['https://data.gesdisc.earthdata.nasa.gov/data/GPM_L3/GPM_3IMERGDF.07/2018/07/3B-DAY.MS.MRG.3IMERG.20180701-S000000-E235959.V07B.nc4']
1 Collection: {'ShortName': 'GPM_3IMERGDF', 'Version': '07'}
Spatial coverage: {'HorizontalSpatialDomain': {'Geometry': {'BoundingRectangles': [{'WestBoundingCoordinate': -180.0, 'EastBoundingCoordinate': 180.0, 'NorthBoundingCoordinate': 90.0, 'SouthBoundingCoordinate': -90.0}]}}}
Temporal coverage: {'RangeDateTime': {'BeginningDateTime': '2018-07-02T00:00:00.000Z', 'EndingDateTime': '20

In [8]:
downloaded_imerg = earthaccess.download(
    imerg_results,
    local_path=str(IMERG_PILOT)
)

print("Downloaded files:", len(downloaded_imerg))

for file in downloaded_imerg:
    print(file)

QUEUEING TASKS | :   0%|          | 0/7 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/7 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/7 [00:00<?, ?it/s]

Downloaded files: 7
Z:\Projects\monsoon-postprocessing\data\raw\imerg_pilot\3B-DAY.MS.MRG.3IMERG.20180701-S000000-E235959.V07B.nc4
Z:\Projects\monsoon-postprocessing\data\raw\imerg_pilot\3B-DAY.MS.MRG.3IMERG.20180702-S000000-E235959.V07B.nc4
Z:\Projects\monsoon-postprocessing\data\raw\imerg_pilot\3B-DAY.MS.MRG.3IMERG.20180703-S000000-E235959.V07B.nc4
Z:\Projects\monsoon-postprocessing\data\raw\imerg_pilot\3B-DAY.MS.MRG.3IMERG.20180704-S000000-E235959.V07B.nc4
Z:\Projects\monsoon-postprocessing\data\raw\imerg_pilot\3B-DAY.MS.MRG.3IMERG.20180705-S000000-E235959.V07B.nc4
Z:\Projects\monsoon-postprocessing\data\raw\imerg_pilot\3B-DAY.MS.MRG.3IMERG.20180706-S000000-E235959.V07B.nc4
Z:\Projects\monsoon-postprocessing\data\raw\imerg_pilot\3B-DAY.MS.MRG.3IMERG.20180707-S000000-E235959.V07B.nc4


In [9]:
imerg_files = sorted([
    file
    for file in IMERG_PILOT.iterdir()
    if file.suffix.lower() in {
        ".nc",
        ".nc4",
        ".h5",
        ".hdf5"
    }
])

print("IMERG files available:", len(imerg_files))

for file in imerg_files:
    print(file.name)

IMERG files available: 7
3B-DAY.MS.MRG.3IMERG.20180701-S000000-E235959.V07B.nc4
3B-DAY.MS.MRG.3IMERG.20180702-S000000-E235959.V07B.nc4
3B-DAY.MS.MRG.3IMERG.20180703-S000000-E235959.V07B.nc4
3B-DAY.MS.MRG.3IMERG.20180704-S000000-E235959.V07B.nc4
3B-DAY.MS.MRG.3IMERG.20180705-S000000-E235959.V07B.nc4
3B-DAY.MS.MRG.3IMERG.20180706-S000000-E235959.V07B.nc4
3B-DAY.MS.MRG.3IMERG.20180707-S000000-E235959.V07B.nc4


In [10]:
from datetime import timedelta

import pandas as pd
import requests


# IMERG observation dates.
observation_dates = pd.date_range(
    start="2018-07-01",
    end="2018-07-07",
    freq="D"
)

GEFS_BASE_URL = (
    "https://noaa-gefs-retrospective.s3.amazonaws.com"
)


def build_gefs_url(observation_date):
    """Return the GEFS file URL initialized one day earlier."""

    initialization_date = (
        observation_date - timedelta(days=1)
    )

    initialization = initialization_date.strftime(
        "%Y%m%d00"
    )

    year = initialization_date.strftime("%Y")

    filename = (
        f"apcp_sfc_{initialization}_c00.grib2"
    )

    url = (
        f"{GEFS_BASE_URL}/"
        f"GEFSv12/reforecast/{year}/"
        f"{initialization}/c00/Days:1-10/"
        f"{filename}"
    )

    return initialization_date, filename, url

In [11]:
download_plan = []

for observation_date in observation_dates:
    initialization_date, filename, url = (
        build_gefs_url(observation_date)
    )

    download_plan.append({
        "observation_date": observation_date.date(),
        "initialization_date": initialization_date.date(),
        "filename": filename,
        "url": url
    })


download_plan_df = pd.DataFrame(download_plan)
download_plan_df

,observation_date,initialization_date,filename,url
0,2018-07-01,2018-06-30,apcp_sfc_2018063000_c00.grib2,https://noaa-gefs-retrospective.s3.amazonaws.c...
1,2018-07-02,2018-07-01,apcp_sfc_2018070100_c00.grib2,https://noaa-gefs-retrospective.s3.amazonaws.c...
2,2018-07-03,2018-07-02,apcp_sfc_2018070200_c00.grib2,https://noaa-gefs-retrospective.s3.amazonaws.c...
3,2018-07-04,2018-07-03,apcp_sfc_2018070300_c00.grib2,https://noaa-gefs-retrospective.s3.amazonaws.c...
4,2018-07-05,2018-07-04,apcp_sfc_2018070400_c00.grib2,https://noaa-gefs-retrospective.s3.amazonaws.c...
5,2018-07-06,2018-07-05,apcp_sfc_2018070500_c00.grib2,https://noaa-gefs-retrospective.s3.amazonaws.c...
6,2018-07-07,2018-07-06,apcp_sfc_2018070600_c00.grib2,https://noaa-gefs-retrospective.s3.amazonaws.c...


In [12]:
def download_file(url, destination):
    """Download one file without loading it fully into memory."""

    # Skip a valid existing download.
    if destination.exists() and destination.stat().st_size > 0:
        print("Already available:", destination.name)
        return True

    temporary_file = destination.with_suffix(
        destination.suffix + ".part"
    )

    try:
        with requests.get(
            url,
            stream=True,
            timeout=(30, 300)
        ) as response:
            response.raise_for_status()

            expected_size = int(
                response.headers.get(
                    "content-length",
                    0
                )
            )

            with open(temporary_file, "wb") as output:
                for chunk in response.iter_content(
                    chunk_size=1024 * 1024
                ):
                    if chunk:
                        output.write(chunk)

        downloaded_size = temporary_file.stat().st_size

        if expected_size and downloaded_size != expected_size:
            raise IOError(
                "Downloaded size does not match "
                "the server file size."
            )

        temporary_file.replace(destination)

        print(
            f"Downloaded: {destination.name} "
            f"({downloaded_size / 1_000_000:.1f} MB)"
        )

        return True

    except Exception as error:
        print("Download failed:", destination.name)
        print("Reason:", error)

        if temporary_file.exists():
            temporary_file.unlink()

        return False

In [13]:
download_results = []

for observation_date in observation_dates:
    initialization_date, filename, url = (
        build_gefs_url(observation_date)
    )

    destination = GEFS_PILOT / filename

    success = download_file(
        url,
        destination
    )

    download_results.append({
        "observation_date": observation_date.date(),
        "initialization_date": initialization_date.date(),
        "filename": filename,
        "success": success
    })

Downloaded: apcp_sfc_2018063000_c00.grib2 (29.6 MB)
Downloaded: apcp_sfc_2018070100_c00.grib2 (26.2 MB)
Downloaded: apcp_sfc_2018070200_c00.grib2 (27.3 MB)
Downloaded: apcp_sfc_2018070300_c00.grib2 (27.6 MB)
Downloaded: apcp_sfc_2018070400_c00.grib2 (29.1 MB)
Downloaded: apcp_sfc_2018070500_c00.grib2 (29.0 MB)
Downloaded: apcp_sfc_2018070600_c00.grib2 (27.9 MB)


In [14]:
download_results_df = pd.DataFrame(
    download_results
)

download_results_df

,observation_date,initialization_date,filename,success
0,2018-07-01,2018-06-30,apcp_sfc_2018063000_c00.grib2,True
1,2018-07-02,2018-07-01,apcp_sfc_2018070100_c00.grib2,True
2,2018-07-03,2018-07-02,apcp_sfc_2018070200_c00.grib2,True
3,2018-07-04,2018-07-03,apcp_sfc_2018070300_c00.grib2,True
4,2018-07-05,2018-07-04,apcp_sfc_2018070400_c00.grib2,True
5,2018-07-06,2018-07-05,apcp_sfc_2018070500_c00.grib2,True
6,2018-07-07,2018-07-06,apcp_sfc_2018070600_c00.grib2,True


In [15]:
successful_downloads = int(
    download_results_df["success"].sum()
)

print(
    "Successful GEFS downloads:",
    successful_downloads,
    "/",
    len(download_results_df)
)

if successful_downloads != len(download_results_df):
    print("Some downloads failed. Rerun Cell 10.")
else:
    print("All matching GEFS files are ready.")

Successful GEFS downloads: 7 / 7
All matching GEFS files are ready.


In [16]:
gefs_files = sorted(
    GEFS_PILOT.glob("*.grib2")
)

print("GEFS files available:", len(gefs_files))

for file in gefs_files:
    print(
        file.name,
        f"{file.stat().st_size / 1_000_000:.1f} MB"
    )

GEFS files available: 7
apcp_sfc_2018063000_c00.grib2 29.6 MB
apcp_sfc_2018070100_c00.grib2 26.2 MB
apcp_sfc_2018070200_c00.grib2 27.3 MB
apcp_sfc_2018070300_c00.grib2 27.6 MB
apcp_sfc_2018070400_c00.grib2 29.1 MB
apcp_sfc_2018070500_c00.grib2 29.0 MB
apcp_sfc_2018070600_c00.grib2 27.9 MB
